In [0]:
# CELDA 0 — Imports
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, IntegerType
)
from pyspark.sql.functions import (
    col, trim, lower, to_date, when, coalesce,
    current_timestamp, lit, year, month,
    sum as _sum, count, avg, round as _round
)
from delta.tables import DeltaTable

print("Librerías importadas ✓")


Librerías importadas ✓


In [0]:
spark.conf.set(
    "fs.azure.account.auth.type.stadatatransandino.dfs.core.windows.net",
    "SharedKey"
)

spark.conf.set(
    "fs.azure.account.key.stadatatransandino.dfs.core.windows.net",
    "C9B6eBqbzAAx6T7pPCzRZHR+QU+ICq9Cxjt/k/iy9umR+Rl+yCu5uNVOlp0f3uDYKtftDo4igTto+ASt3UqZhw=="
)

dbutils.fs.ls("abfss://landing@stadatatransandino.dfs.core.windows.net/")

[FileInfo(path='abfss://landing@stadatatransandino.dfs.core.windows.net/clientes/', name='clientes/', size=0, modificationTime=1781999938000),
 FileInfo(path='abfss://landing@stadatatransandino.dfs.core.windows.net/envios/', name='envios/', size=0, modificationTime=1781999945000),
 FileInfo(path='abfss://landing@stadatatransandino.dfs.core.windows.net/incidencias_23/', name='incidencias_23/', size=0, modificationTime=1781999938000),
 FileInfo(path='abfss://landing@stadatatransandino.dfs.core.windows.net/incidencias_24/', name='incidencias_24/', size=0, modificationTime=1781999937000),
 FileInfo(path='abfss://landing@stadatatransandino.dfs.core.windows.net/rutas/', name='rutas/', size=0, modificationTime=1781999942000),
 FileInfo(path='abfss://landing@stadatatransandino.dfs.core.windows.net/transportistas/', name='transportistas/', size=0, modificationTime=1781999938000)]

In [0]:
# CELDA 1 — Parámetro de ADF y verificación inicial
dbutils.widgets.text("fecha_proceso", "2024-01-15")
fecha_proceso = dbutils.widgets.get("fecha_proceso")
print(f"fecha_proceso recibido: {fecha_proceso}")

# Verificar catálogo
spark.sql("USE CATALOG dbw_transandino_v2")
spark.sql("USE SCHEMA default")
print("Catálogo y schema accesibles ✓")

# Verificar containers ADLS
STORAGE = "stadatatransandino"
containers = ["landing", "bronze", "silver", "gold"]
for c in containers:
    path = f"abfss://{c}@{STORAGE}.dfs.core.windows.net/"
    try:
        dbutils.fs.ls(path)
        print(f"Container {c}/ accesible ✓")
    except:
        print(f"Container {c}/ NO accesible ✗")

fecha_proceso recibido: 2024-01-15
Catálogo y schema accesibles ✓
Container landing/ accesible ✓
Container bronze/ accesible ✓
Container silver/ accesible ✓
Container gold/ accesible ✓


In [0]:
# CELDA 2 — Schema definitions
from datetime import datetime

STORAGE = "stadatatransandino"
BASE_PATH = f"abfss://{{container}}@{STORAGE}.dfs.core.windows.net/{{tabla}}"
LANDING = f"abfss://landing@{STORAGE}.dfs.core.windows.net"

def adls_path(container, tabla):
    return f"abfss://{container}@{STORAGE}.dfs.core.windows.net/{tabla}"

# Schemas Bronze — todos StringType
schema_envios = StructType([
    StructField("id_envio",         StringType(), True),
    StructField("id_cliente",       StringType(), True),
    StructField("id_transportista", StringType(), True),
    StructField("id_ruta",          StringType(), True),
    StructField("peso_kg",          StringType(), True),
    StructField("monto_flete",      StringType(), True),
    StructField("fecha_envio",      StringType(), True),
    StructField("estado",           StringType(), True),
    StructField("updated_at",       StringType(), True),
])

schema_transportistas = StructType([
    StructField("id_transportista", StringType(), True),
    StructField("nombre",           StringType(), True),
    StructField("zona",             StringType(), True),
    StructField("ciudad",           StringType(), True),
    StructField("modalidad",        StringType(), True),
])

schema_rutas = StructType([
    StructField("id_ruta",       StringType(), True),
    StructField("origen",        StringType(), True),
    StructField("destino",       StringType(), True),
    StructField("tipo_ruta",     StringType(), True),
    StructField("distancia_km",  StringType(), True),
])

schema_clientes = StructType([
    StructField("id_cliente",  StringType(), True),
    StructField("nombre",      StringType(), True),
    StructField("segmento",    StringType(), True),
    StructField("ciudad",      StringType(), True),
    StructField("fecha_alta",  StringType(), True),
])

schema_incidencias = StructType([
    StructField("id_incidencia",      StringType(), True),
    StructField("id_envio",           StringType(), True),
    StructField("tipo_incidencia",    StringType(), True),
    StructField("fecha_incidencia",   StringType(), True),
    StructField("resolucion",         StringType(), True),
    StructField("estado_resolucion",  StringType(), True),
])

print("Schemas definidos ✓")

Schemas definidos ✓


In [0]:
# CELDA 3 — Capa Bronze
tablas_bronze = [
    ("envios",            schema_envios,         "envios"),
    ("transportistas",    schema_transportistas,  "transportistas"),
    ("rutas",             schema_rutas,           "rutas"),
    ("clientes",            schema_clientes,        "clientes"),
    ("incidencias_23",  schema_incidencias,     "incidencias"),
]

for archivo, schema, nombre_tabla in tablas_bronze:
    # Leer CSV desde landing usando la fecha_proceso
    ruta_csv = f"{LANDING}/{archivo}/{fecha_proceso}/{archivo}.csv"
    
    df = (spark.read
          .option("header", "true")
          .option("sep", ",")
          .schema(schema)
          .csv(ruta_csv))
    
    # Agregar columna de ingesta
    df = df.withColumn("ingestion_timestamp", current_timestamp())
    
    # Escribir Delta en bronze/
    ruta_delta = adls_path("bronze", nombre_tabla)
    (df.write
       .format("delta")
       .mode("overwrite")
       .save(ruta_delta))
    
    # Registrar en Unity Catalog
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.bronze_{nombre_tabla}
        USING DELTA
        LOCATION '{ruta_delta}'
    """)
    
total_filas = spark.read.format("delta").load(ruta_delta).count()
print(f"bronze_{nombre_tabla}: {total_filas} filas ✓")

bronze_incidencias: 120 filas ✓


In [0]:
# CELDA 4 — Capa Silver
# --- silver_envios ---
df_env = spark.table("dbw_transandino_v2.default.bronze_envios")

# Resolver los 5 formatos de fecha
df_env = df_env.withColumn("fecha_envio",
    coalesce(
        to_date(col("fecha_envio"), "yyyy-MM-dd"),
        to_date(col("fecha_envio"), "MM-dd-yyyy"),
        to_date(col("fecha_envio"), "dd/MM/yyyy"),
        to_date(col("fecha_envio"), "yyyy/MM/dd"),
        to_date(col("fecha_envio"), "yyyy/MM/dd HH:mm:ss"),
    )
)

# Numéricos — N/A a null
df_env = (df_env
    .withColumn("peso_kg",
        when(trim(col("peso_kg")) == "N/A", None)
        .otherwise(col("peso_kg").cast(DoubleType())))
    .withColumn("monto_flete",
        when(trim(col("monto_flete")) == "N/A", None)
        .otherwise(col("monto_flete").cast(DoubleType())))
    .withColumn("id_ruta",
        when(trim(col("id_ruta")) == "N/A", None)
        .otherwise(col("id_ruta")))
)

# Normalizar estado
df_env = df_env.withColumn("estado", lower(trim(col("estado"))))

# Eliminar duplicados
df_env = df_env.dropDuplicates(["id_envio"])

# Escribir Silver
ruta_silver_envios = adls_path("silver", "envios")
(df_env.write.format("delta").mode("overwrite").save(ruta_silver_envios))
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.silver_envios
    USING DELTA LOCATION '{ruta_silver_envios}'
""")
print(f"silver_envios: {df_env.count()} filas ✓")

# --- silver_transportistas ---
df_t = spark.table("dbw_transandino_v2.default.bronze_transportistas")
df_t = df_t.dropDuplicates(["id_transportista"])
ruta = adls_path("silver", "transportistas")
df_t.write.format("delta").mode("overwrite").save(ruta)
spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.silver_transportistas USING DELTA LOCATION '{ruta}'")
print(f"silver_transportistas: {df_t.count()} filas ✓")

# --- silver_rutas ---
df_r = spark.table("dbw_transandino_v2.default.bronze_rutas")
df_r = df_r.withColumn("distancia_km", col("distancia_km").cast(DoubleType()))
df_r = df_r.dropDuplicates(["id_ruta"])
ruta = adls_path("silver", "rutas")
df_r.write.format("delta").mode("overwrite").save(ruta)
spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.silver_rutas USING DELTA LOCATION '{ruta}'")
print(f"silver_rutas: {df_r.count()} filas ✓")

# --- silver_clientes ---
df_c = spark.table("dbw_transandino_v2.default.bronze_clientes")
df_c = df_c.withColumn("fecha_alta", to_date(col("fecha_alta"), "yyyy-MM-dd"))
df_c = df_c.dropDuplicates(["id_cliente"])
ruta = adls_path("silver", "clientes")
df_c.write.format("delta").mode("overwrite").save(ruta)
spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.silver_clientes USING DELTA LOCATION '{ruta}'")
print(f"silver_clientes: {df_c.count()} filas ✓")

# --- silver_incidencias (solo 2023 en esta etapa) ---
df_i = spark.table("dbw_transandino_v2.default.bronze_incidencias")
df_i = df_i.withColumn("fecha_incidencia", to_date(col("fecha_incidencia"), "yyyy-MM-dd"))
df_i = df_i.dropDuplicates(["id_incidencia"])
ruta = adls_path("silver", "incidencias")
df_i.write.format("delta").mode("overwrite").save(ruta)
spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.silver_incidencias USING DELTA LOCATION '{ruta}'")
print(f"silver_incidencias: {df_i.count()} filas ✓")

silver_envios: 600 filas ✓
silver_transportistas: 20 filas ✓
silver_rutas: 30 filas ✓
silver_clientes: 60 filas ✓
silver_incidencias: 120 filas ✓


In [0]:
# CELDA 5 — Carga incremental incidencias_2024 con MERGE INTO

# Leer incidencias_2024 desde landing
ruta_csv_2024 = f"{LANDING}/incidencias_24/{fecha_proceso}/incidencias_24.csv"
df_2024 = (spark.read
           .option("header", "true")
           .schema(schema_incidencias)
           .csv(ruta_csv_2024))
df_2024 = (df_2024
    .withColumn("fecha_incidencia", to_date(col("fecha_incidencia"), "yyyy-MM-dd"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

# MERGE INTO silver_incidencias
ruta_silver_inc = adls_path("silver", "incidencias")
delta_table = DeltaTable.forPath(spark, ruta_silver_inc)

delta_table.alias("target").merge(
    df_2024.alias("source"),
    "target.id_incidencia = source.id_incidencia"
).whenMatchedUpdate(set={
    "estado_resolucion": "source.estado_resolucion",
    "resolucion":        "source.resolucion"
}).whenNotMatchedInsertAll(
).execute()

print("MERGE completado ✓")

# Verificar con Delta Time Travel
spark.sql(f"""
    SELECT COUNT(*) as filas_version_anterior
    FROM delta.`{ruta_silver_inc}` VERSION AS OF 0
""").show()

spark.sql(f"""
    SELECT COUNT(*) as filas_version_actual
    FROM delta.`{ruta_silver_inc}`
""").show()

MERGE completado ✓
+----------------------+
|filas_version_anterior|
+----------------------+
|                   120|
+----------------------+

+--------------------+
|filas_version_actual|
+--------------------+
|                 170|
+--------------------+



In [0]:
# CELDA 6 — Capa Gold

df_env_s   = spark.table("dbw_transandino_v2.default.silver_envios")
df_trans_s = spark.table("dbw_transandino_v2.default.silver_transportistas")
df_rutas_s = spark.table("dbw_transandino_v2.default.silver_rutas")

# --- gold_envios_zona ---
df_zona = (df_env_s
    .join(df_trans_s, "id_transportista", "left")
    .withColumn("anio", year("fecha_envio"))
    .withColumn("mes",  month("fecha_envio"))
    .groupBy("zona", "anio", "mes")
    .agg(
        count("id_envio").alias("total_envios"),
        _round(_sum("monto_flete"), 2).alias("monto_total"),
        _round(_sum("peso_kg"), 2).alias("peso_total"),
        _round(_sum("monto_flete") / count("id_envio"), 2).alias("ticket_promedio")
    )
)

ruta_gold_zona = adls_path("gold", "envios_zona")
df_zona.write.format("delta").mode("overwrite").save(ruta_gold_zona)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.gold_envios_zona
    USING DELTA LOCATION '{ruta_gold_zona}'
""")
print(f"gold_envios_zona: {df_zona.count()} filas ✓")

# --- gold_envios_tipo_ruta ---
df_tipo = (df_env_s
    .join(df_rutas_s, "id_ruta", "left")
    .withColumn("anio", year("fecha_envio"))
    .withColumn("mes",  month("fecha_envio"))
    .groupBy("tipo_ruta", "anio", "mes")
    .agg(
        count("id_envio").alias("total_envios"),
        _round(_sum("monto_flete"), 2).alias("monto_total"),
        _round(_sum("monto_flete") / count("id_envio"), 2).alias("ticket_promedio"),
        _round(avg("distancia_km"), 2).alias("distancia_promedio")
    )
)

ruta_gold_tipo = adls_path("gold", "envios_tipo_ruta")
df_tipo.write.format("delta").mode("overwrite").save(ruta_gold_tipo)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.gold_envios_tipo_ruta
    USING DELTA LOCATION '{ruta_gold_tipo}'
""")
print(f"gold_envios_tipo_ruta: {df_tipo.count()} filas ✓")

# --- Verificación cruzada ---
total_silver = spark.table("dbw_transandino_v2.default.silver_envios").count()
total_gold   = spark.sql("SELECT SUM(total_envios) as total FROM dbw_transandino_v2.default.gold_envios_zona").collect()[0]["total"]
diferencia   = abs(total_silver - total_gold)
print(f"silver_envios:    {total_silver} filas")
print(f"gold_envios_zona: {total_gold} envíos sumados")
print(f"Diferencia:       {diferencia} {'✓ dentro de tolerancia' if diferencia <= 10 else '✗ revisar joins'}")

gold_envios_zona: 151 filas ✓
gold_envios_tipo_ruta: 108 filas ✓
silver_envios:    600 filas
gold_envios_zona: 600 envíos sumados
Diferencia:       0 ✓ dentro de tolerancia


In [0]:
from pyspark.sql.functions import make_date, lit, col

STORAGE = "stadatatransandino"
KEY = "C9B6eBqbzAAx6T7pPCzRZHR+QU+ICq9Cxjt/k/iy9umR+Rl+yCu5uNVOlp0f3uDYKtftDo4igTto+ASt3UqZhw=="

spark.conf.set(f"fs.azure.account.auth.type.{STORAGE}.dfs.core.windows.net", "SharedKey")
spark.conf.set(f"fs.azure.account.key.{STORAGE}.dfs.core.windows.net", KEY)

# Verificar acceso primero
dbutils.fs.ls(f"abfss://gold@{STORAGE}.dfs.core.windows.net/")

from pyspark.sql.functions import make_date, lit, col

STORAGE = "stadatatransandino"
KEY = "C9B6eBqbzAAx6T7pPCzRZHR+QU+ICq9Cxjt/k/iy9umR+Rl+yCu5uNVOlp0f3uDYKtftDo4igTto+ASt3UqZhw=="

spark.conf.set(f"fs.azure.account.auth.type.{STORAGE}.dfs.core.windows.net", "SharedKey")
spark.conf.set(f"fs.azure.account.key.{STORAGE}.dfs.core.windows.net", KEY)

ruta = f"abfss://gold@{STORAGE}.dfs.core.windows.net/envios_zona_mes"

df_zona_mes = (spark.table("dbw_transandino_v2.default.gold_envios_zona")
    .withColumn("fecha_mes", make_date(col("anio").cast("int"), col("mes").cast("int"), lit(1))))

df_zona_mes.write.format("delta").mode("overwrite").save(ruta)

spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_transandino_v2.default.gold_envios_zona_mes USING DELTA LOCATION '{ruta}'")

print(f"gold_envios_zona_mes: {df_zona_mes.count()} filas ✓")

gold_envios_zona_mes: 151 filas ✓


In [0]:
# CELDA 7 — Limpieza (comentado por defecto — ejecutar solo al finalizar el TP)

# tablas = [
#     "bronze_envios", "bronze_transportistas", "bronze_rutas",
#     "bronze_clientes", "bronze_incidencias",
#     "silver_envios", "silver_transportistas", "silver_rutas",
#     "silver_clientes", "silver_incidencias",
#     "gold_envios_zona", "gold_envios_tipo_ruta"
# ]
# for tabla in tablas:
#     spark.sql(f"DROP TABLE IF EXISTS azure_de.default.{tabla}")
#     print(f"Tabla {tabla} eliminada")

# containers = ["bronze", "silver", "gold", "landing"]
# for c in containers:
#     dbutils.fs.rm(f"abfss://{c}@stadatatransandino.dfs.core.windows.net/", recurse=True)
#     print(f"Container {c}/ limpiado")

print("Celda de limpieza lista (comentada) ✓")

Celda de limpieza lista (comentada) ✓
